## 🎯 Learning Objectives
* Understand the concept and necessity of sub-question decomposition in advanced RAG systems.
* Learn how to implement sub-question decomposition using LlamaIndex's `SubQuestionQueryEngine`.
* Analyze the performance implications, benefits, and trade-offs of using sub-question decomposition.
* Identify suitable use cases for applying sub-question decomposition in production RAG applications.


## Sub-question Decomposition: Tackling Complex Queries in RAG

In the realm of Retrieval Augmented Generation (RAG), a common challenge arises when users pose complex, multi-faceted, or multi-hop questions. A standard RAG pipeline, which typically retrieves documents based on a single query and then synthesizes an answer, often struggles with these types of questions. It might retrieve irrelevant information, miss crucial details, or even hallucinate if the context isn't perfectly aligned with the intricate nature of the query.

This is where **Sub-question Decomposition** comes into play. It's an advanced RAG pattern designed to break down a complex user query into a series of simpler, atomic sub-questions. Each sub-question can then be answered independently using a standard RAG approach, and their individual answers are finally synthesized to form a comprehensive response to the original complex query.

### The Analogy: Deconstructing a Research Project

Imagine you're a research assistant tasked with answering a complex question like: "*What were the key findings of the latest climate change report regarding sea-level rise, and how do these findings compare with the projections from the previous decade's report? Also, what policy recommendations were made to address these changes?*"

Trying to answer this all at once would be overwhelming. Instead, a smart research assistant would naturally break it down:

1.  **Sub-question 1:** "What were the key findings of the latest climate change report regarding sea-level rise?"
2.  **Sub-question 2:** "What were the sea-level rise projections from the previous decade's climate change report?"
3.  **Sub-question 3:** "How do the findings from the latest report compare with the projections from the previous decade's report?"
4.  **Sub-question 4:** "What policy recommendations were made in the latest report to address sea-level rise?"

Each sub-question is simpler, allowing the assistant to focus on retrieving specific information for each. Once all sub-questions are answered, the assistant synthesizes these individual answers into a coherent, comprehensive response to the original complex query.

### How Sub-question Decomposition Works in RAG:

1.  **Query Reception:** The user submits a complex query.
2.  **Decomposition (LLM-powered):** A powerful Large Language Model (LLM) acts as the orchestrator. It analyzes the complex query and, based on its understanding, breaks it down into a list of simpler, independent sub-questions. This step often involves prompt engineering to guide the LLM effectively.
3.  **Sub-question Execution (RAG Calls):** Each generated sub-question is then fed into a dedicated RAG pipeline (or a specific `QueryEngineTool` in LlamaIndex). This means for each sub-question, relevant documents are retrieved, and an answer is generated.
4.  **Answer Synthesis (LLM-powered):** Once all sub-questions have been answered, the original LLM (or another specialized LLM) takes all the individual sub-answers and the original complex query. It then synthesizes these pieces of information into a single, coherent, and comprehensive final answer.

### Benefits:

*   **Improved Accuracy:** By focusing RAG on smaller, more specific questions, the system is less likely to retrieve irrelevant context or hallucinate.
*   **Multi-hop Reasoning:** Effectively handles questions that require combining information from multiple distinct pieces of knowledge or documents.
*   **Better Context Management:** Prevents overwhelming the RAG's context window with too much information for a single, broad query.
*   **Explainability (Potential):** The intermediate sub-questions and their answers can sometimes offer insights into the reasoning process.

### Challenges:

*   **Increased Latency:** Multiple LLM calls (for decomposition and synthesis) and multiple RAG calls (for each sub-question) significantly increase the overall response time.
*   **Higher Cost:** More LLM token usage translates to higher operational costs.
*   **Error Propagation:** Errors in the decomposition step (e.g., generating irrelevant sub-questions) or in the synthesis step can lead to incorrect final answers.
*   **Orchestration Complexity:** Requires careful management of the workflow and robust LLM prompting.


In [ ]:
# Install necessary libraries (as of 2026, these are stable and widely used)
# !pip install llama-index openai pypdf

import os
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, ServiceContext
from llama_index.core.llms import LLM
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core.query_engine import SubQuestionQueryEngine
from llama_index.core.tools import QueryEngineTool, ToolMetadata
from llama_index.core.callbacks import CallbackManager, LlamaDebugHandler

# --- Configuration --- 
# Set your OpenAI API key. In a production environment, use environment variables or a secure secret manager.
# os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"

# For demonstration purposes, we'll use a placeholder if the key isn't set.
# In a real scenario, ensure this is properly configured.
if "OPENAI_API_KEY" not in os.environ:
    print("Warning: OPENAI_API_KEY not found. Using a placeholder. Please set your API key for actual execution.")
    # This will likely fail without a real key, but allows the code structure to be seen.
    os.environ["OPENAI_API_KEY"] = "sk-xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"

# Configure LLM and Embedding Model
# Using GPT-4o for its strong reasoning capabilities for decomposition and synthesis
# and text-embedding-3-large for high-quality embeddings.
llm = OpenAI(model="gpt-4o", temperature=0.1)
embed_model = OpenAIEmbedding(model="text-embedding-3-large")

# Setup LlamaDebugHandler to observe the internal workings of the query engine
llama_debug = LlamaDebugHandler(print_trace_on_end=True)
callback_manager = CallbackManager([llama_debug])

# Create a ServiceContext
service_context = ServiceContext.from_defaults(
    llm=llm,
    embed_model=embed_model,
    callback_manager=callback_manager
)

# --- Data Loading and Indexing --- 
# For this example, we'll create synthetic documents to simulate a knowledge base.
# In a real application, you would load documents from files (PDFs, text, etc.).

from llama_index.core.schema import Document

documents = [
    Document(text="""
    The 2023 Intergovernmental Panel on Climate Change (IPCC) report, titled 'Climate Change 2023: Synthesis Report',
    highlighted that global mean sea level (GMSL) has risen faster since 1900 than over any preceding century in at least the last 3,000 years.
    The report projects that under a very high emissions scenario (SSP5-8.5), GMSL is likely to rise by 0.63 to 1.01 meters by 2100 relative to 1995-2014.
    Key findings include accelerated ice sheet melt in Greenland and Antarctica, and thermal expansion of ocean waters.
    Policy recommendations include rapid and deep cuts in greenhouse gas emissions, investing in renewable energy, protecting and restoring ecosystems,
    and developing early warning systems for coastal communities. The report emphasizes the urgency of adaptation measures.
    """),
    Document(text="""
    The 2013 IPCC Fifth Assessment Report (AR5), Working Group I, projected global mean sea level rise for 2081-2100 relative to 1986-2005.
    Under a high emissions scenario (RCP8.5), the projection was for a rise of 0.45 to 0.82 meters. Under a low emissions scenario (RCP2.6),
    it was 0.26 to 0.55 meters. AR5 noted that the rate of GMSL rise since the mid-19th century has been larger than the mean rate during the previous two millennia.
    It attributed sea level rise primarily to thermal expansion and glacier melt. AR5 did not provide detailed policy recommendations in the same manner as the 2023 synthesis report,
    focusing more on scientific assessments of climate change.
    """),
    Document(text="""
    A recent study published in 'Nature Climate Change' in late 2025 indicated that current sea-level rise models might be underestimating the contribution
    from Antarctic ice sheet instability, suggesting a potential for an additional 10-20 cm rise by 2100 under worst-case scenarios.
    This study calls for updated policy frameworks to account for these new projections.
    """)
]

# Create a VectorStoreIndex from the documents
index = VectorStoreIndex.from_documents(documents, service_context=service_context)

# --- Define Query Engine Tools --- 
# We'll create a single query engine tool that can answer questions about climate reports.
# In a more complex scenario, you might have multiple tools for different data sources/domains.

climate_report_engine = index.as_query_engine(similarity_top_k=3)

query_engine_tools = [
    QueryEngineTool(
        query_engine=climate_report_engine,
        metadata=ToolMetadata(
            name="climate_report_data",
            description=(
                "Provides information about climate change reports, including sea-level rise projections, key findings, and policy recommendations."
            ),
        ),
    ),
]

# --- Initialize SubQuestionQueryEngine --- 
# This is the core component for decomposition.
# It uses the LLM to break down the query and then uses the provided tools to answer sub-questions.

sub_question_engine = SubQuestionQueryEngine.from_defaults(
    query_engine_tools=query_engine_tools,
    service_context=service_context,
    verbose=True # Set to True to see the sub-questions generated and their individual answers
)

# --- Execute a Complex Query --- 
complex_query = (
    "What were the key findings of the 2023 IPCC report regarding sea-level rise, "
    "how do these compare with the 2013 IPCC report's projections, "
    "and what policy recommendations were made in the 2023 report?"
)

print(f"\n--- Original Complex Query ---\n{complex_query}\n")
response = sub_question_engine.query(complex_query)

print(f"\n--- Final Answer ---\n{response.response}\n")

# You can also inspect the source nodes and intermediate steps if verbose is True
# print(response.source_nodes)
# print(response.metadata)


### Interpreting the Output and Performance Considerations

When you run the code, observe the `verbose=True` output from the `SubQuestionQueryEngine`. You'll see several key stages:

1.  **Sub-question Generation:** The LLM first takes the `complex_query` and generates a list of simpler, atomic sub-questions. For our example query, you should see something similar to:
    *   "What were the key findings of the 2023 IPCC report regarding sea-level rise?"
    *   "What were the sea-level rise projections from the 2013 IPCC report?"
    *   "How do the 2023 IPCC report's findings on sea-level rise compare with the 2013 IPCC report's projections?"
    *   "What policy recommendations were made in the 2023 IPCC report?"

2.  **Tool Selection and Execution:** For each sub-question, the `SubQuestionQueryEngine` identifies the most appropriate `QueryEngineTool` (in our case, the `climate_report_data` tool) and executes the sub-question against it. You'll see the individual answers generated for each sub-question.

3.  **Synthesis:** Finally, the LLM takes all the individual sub-answers and the original complex query, and synthesizes them into a single, coherent `Final Answer`.

This step-by-step process demonstrates how a seemingly intractable complex query is systematically broken down and addressed.

### Performance Trade-offs and Use Cases

**Benefits:**

*   **Enhanced Accuracy for Complex Queries:** This is the primary benefit. By breaking down queries, the RAG system can retrieve more precise information for each component, leading to a more accurate and comprehensive final answer, especially for multi-hop questions or those requiring comparative analysis.
*   **Reduced Hallucination:** When the RAG system focuses on smaller, well-defined sub-questions, the likelihood of the LLM generating unsupported or incorrect information decreases significantly because the retrieved context is more targeted.
*   **Improved Context Management:** Prevents overloading the LLM's context window with a single, massive retrieval for a broad query. Each sub-query gets its own focused retrieval.

**Drawbacks:**

*   **Increased Latency:** The most significant drawback. Each sub-question requires its own RAG pipeline execution (retrieval + LLM generation), plus the initial decomposition and final synthesis steps, all involving LLM calls. This can lead to significantly longer response times compared to a single-shot RAG query.
*   **Higher Cost:** More LLM calls and potentially more token usage directly translate to higher API costs, especially with advanced models like GPT-4o.
*   **Orchestration Complexity and Fragility:** The quality of the final answer heavily depends on the LLM's ability to accurately decompose the query and then synthesize the sub-answers. Poor decomposition or synthesis can lead to incorrect or incomplete final answers. Careful prompt engineering and robust LLMs are crucial.

**Typical Use Cases (where the benefits outweigh the costs):**

*   **Financial Analysis:** "Compare the Q3 earnings of Company A and Company B, and explain the market's reaction to both." (Requires retrieving two sets of earnings, then comparing, then finding market reaction).
*   **Legal Research:** "Summarize the precedents for intellectual property infringement in software, and how they apply to a new case involving open-source licensing." (Requires identifying precedents, then applying them to a specific scenario).
*   **Scientific Literature Review:** "What are the known side effects of Drug X, and how do they interact with common cardiovascular medications?" (Requires retrieving side effects, then retrieving interactions with a specific class of drugs).
*   **Complex Customer Support:** When a customer asks a question that involves multiple product features, troubleshooting steps, or policy details that are spread across different documentation sections.

Sub-question decomposition is a powerful technique for building highly accurate RAG systems for complex queries, but it should be applied judiciously, considering the trade-offs in latency and cost.


### Resources

*   **LlamaIndex Documentation on Query Engines:** [https://docs.llamaindex.ai/en/stable/module_guides/querying/query_engine/root.html](https://docs.llamaindex.ai/en/stable/module_guides/querying/query_engine/root.html)
*   **LlamaIndex SubQuestionQueryEngine:** [https://docs.llamaindex.ai/en/stable/module_guides/querying/query_engine/sub_question_query_engine.html](https://docs.llamaindex.ai/en/stable/module_guides/querying/query_engine/sub_question_query_engine.html)
*   **OpenAI API Documentation:** [https://platform.openai.com/docs/](https://platform.openai.com/docs/)
*   **Google AI Studio (for Gemini models):** [https://ai.google.dev/](https://ai.google.dev/)
*   **Hugging Face (for open-source LLMs and embeddings):** [https://huggingface.co/](https://huggingface.co/)
